# QRculus · 다변수 QR 판독 성공률 예측 (오류 수정)

Google Colab에서 **런타임 → 모두 실행**을 누르세요. GitHub 원본 데이터를 다운로드하고 **전체 가림률, 세 위치 탐지 패턴의 가림률, 가림 형태, 오류 정정 등급**을 이용해 4가지 모델을 비교합니다.

- `trials.csv`에는 탐지 패턴 가림률이 직접 저장되지 않습니다. 같은 실험의 `mask_orders.csv`에서 가려진 셀 순서를 복원해 직접 계산합니다.
- 위치 탐지 패턴: QR 버전 1의 세 7×7 정사각형, 총 **147칸**. `finder_coverage_pct = 100 × 가린 위치 탐지 칸 / 147`입니다.
- **원래의** `payload_split.csv` 학습 80개/검증 20개를 그대로 사용합니다. 동일한 QR 원문은 학습과 검증에 동시에 사용하지 않습니다.
- Brier score, Log loss, 조건별 성공률 MAE(%p)를 출력합니다. 검증 데이터는 학습에 사용하지 않습니다.

연구 데이터: [QRculus GitHub](https://github.com/hwanghj09/QRculus/tree/main/results/pink_v2).


In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib requests

## 1. 원본 데이터 다운로드 · 탐지 패턴 가림률 계산 · 학습 및 검증

In [ ]:
"""QRculus v2 trial-level multivariate model comparison (Colab).
Uses original train/validation payload split and reconstructs finder coverage
from saved mask orders (same method as src/qr_calculus/pink_analysis.py).
"""
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import log_loss, brier_score_loss
import matplotlib.pyplot as plt

ROOT = 'https://raw.githubusercontent.com/hwanghj09/QRculus/main/results/pink_v2/'
OUT = Path('qrculus_multivariate_results')
OUT.mkdir(parents=True, exist_ok=True)

# Download all three source files: original trials, saved mask order, official split.
files_to_read = {
    'trials.csv': ROOT + 'raw/trials.csv',
    'mask_orders.csv': ROOT + 'raw/mask_orders.csv',
    'payload_split.csv': ROOT + 'payload_split.csv',
}
for filename, url in files_to_read.items():
    path = Path(filename)
    if not path.exists():
        print(f'Downloading {filename} ...')
        pd.read_csv(url, low_memory=False).to_csv(path, index=False)

df = pd.read_csv('trials.csv', low_memory=False)
orders = pd.read_csv('mask_orders.csv', low_memory=False)
splits = pd.read_csv('payload_split.csv')
print('Trial columns:', ', '.join(df.columns))
print(f'{len(df):,} trials, {len(orders):,} mask orders, {len(splits):,} payloads')

required_trials = {'payload_id','repeat_id','mask_type','level','target_pct',
                   'actual_pct','covered_modules','covered_reserved','success'}
required_orders = {'payload_id','repeat_id','mask_type','cells_row_major_index'}
if missing := required_trials.difference(df.columns):
    raise ValueError(f'Missing trial fields: {sorted(missing)}')
if missing := required_orders.difference(orders.columns):
    raise ValueError(f'Missing mask-order fields: {sorted(missing)}')
if not {'payload_id','split'}.issubset(splits.columns):
    raise ValueError('payload_split.csv must contain payload_id and split')

# These exact 7x7 squares are used in the existing QRculus analysis.
SIDE = 21
finder_mask = np.zeros((SIDE, SIDE), dtype=bool)
finder_mask[:7, :7] = True                 # upper left
finder_mask[:7, -7:] = True                # upper right
finder_mask[-7:, :7] = True                # lower left
finder_flat = finder_mask.ravel()
assert finder_flat.sum() == 147

target_values = sorted(int(v) for v in df.loc[df['target_pct'] > 0, 'target_pct'].unique())
ks = {t: round(SIDE * SIDE * t / 100) for t in target_values}
lookup_rows = []
for row in orders.itertuples(index=False):
    order = np.fromstring(row.cells_row_major_index, sep=' ', dtype=np.int16)
    if order.size != SIDE * SIDE or np.unique(order).size != SIDE * SIDE or order.min() < 0 or order.max() >= SIDE * SIDE:
        raise ValueError(f'Invalid mask order for {row.payload_id}/{row.repeat_id}/{row.mask_type}')
    cumulative = np.cumsum(finder_flat[order])
    for target, k in ks.items():
        lookup_rows.append((row.payload_id, int(row.repeat_id), row.mask_type,
                            target, int(cumulative[k-1])))
lookup = pd.DataFrame(lookup_rows, columns=['payload_id','repeat_id','mask_type',
                                             'target_pct','covered_finder'])

df = df.merge(lookup, on=['payload_id','repeat_id','mask_type','target_pct'],
              how='left', validate='many_to_one')
missing_finder = (df['target_pct'] > 0) & df['covered_finder'].isna()
if missing_finder.any():
    raise ValueError(f'{missing_finder.sum():,} nonzero trials do not match saved mask orders')
df['covered_finder'] = df['covered_finder'].fillna(0).astype('int16')
if not (df['covered_finder'] <= df['covered_reserved']).all():
    raise ValueError('Finder coverage is inconsistent with reserved-cell counts')
df['finder_coverage_pct'] = df['covered_finder'] * (100 / 147)

# Use the ORIGINAL split from the experiment (not a new random split).
df = df.merge(splits[['payload_id','split']], on='payload_id', how='left', validate='many_to_one')
if df['split'].isna().any():
    raise ValueError('Some payloads are absent from payload_split.csv')
training = df['split'].eq('train')
validation = df['split'].eq('validation')
if not (training | validation).all() or not training.any() or not validation.any():
    raise ValueError('Expected train/validation labels in payload_split.csv')
print(f'Train {training.sum():,} trials / {df.loc[training,"payload_id"].nunique()} payloads; '
      f'validation {validation.sum():,} trials / {df.loc[validation,"payload_id"].nunique()} payloads')

# Sanity checks: 0% occlusion -> 0% finder occlusion, maximum 100% finder.
assert df.loc[df['covered_modules']==0, 'covered_finder'].eq(0).all()
assert df['finder_coverage_pct'].between(0, 100).all()
X = df[['actual_pct', 'finder_coverage_pct', 'mask_type', 'level']].copy()
X.columns = ['occlusion_pct', 'finder_coverage_pct', 'mask_type', 'ecc_level']
y = pd.to_numeric(df['success'], errors='raise').astype('int8')
assert y.isin([0, 1]).all()

numeric = ['occlusion_pct', 'finder_coverage_pct']
categorical = ['mask_type','ecc_level']
preprocessor = ColumnTransformer([
    ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
])
models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, C=1.0),
    'RandomForest': RandomForestClassifier(n_estimators=120, min_samples_leaf=12,
                                           n_jobs=-1, random_state=2026),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=120, min_samples_leaf=12,
                                       n_jobs=-1, random_state=2026),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=160,
                                                           l2_regularization=3,
                                                           random_state=2026),
}

results, predictions = [], []
for name, estimator in models.items():
    print(f'Training {name} ...')
    pipe = make_pipeline(preprocessor, estimator)
    pipe.fit(X.loc[training], y.loc[training])
    probs = pipe.predict_proba(X.loc[validation])[:, 1]
    results.append({'model': name,
                    'trial_brier': brier_score_loss(y.loc[validation], probs),
                    'trial_log_loss': log_loss(y.loc[validation], probs)})
    z = X.loc[validation].copy()
    z['actual'] = y.loc[validation].to_numpy()
    z['predicted'] = probs
    z['model'] = name
    z['payload_id'] = df.loc[validation, 'payload_id'].to_numpy()
    predictions.append(z)

metrics = pd.DataFrame(results).sort_values('trial_brier')
metrics.to_csv(OUT/'metrics.csv', index=False)
pred = pd.concat(predictions, ignore_index=True)
pred.to_csv(OUT/'predictions.csv', index=False)
print('\nTrial-level evaluation (lower is better):\n', metrics.to_string(index=False))

# Percentage-point errors of success probabilities aggregated by experimental condition.
# These groups use only coverage / shape / ECC so comparisons match prior study.
groupcols = ['occlusion_pct','mask_type','ecc_level']
grp = pred.groupby(['model'] + groupcols, as_index=False)[['actual','predicted']].mean()
grp['abs_error_pp'] = 100 * (grp['actual'] - grp['predicted']).abs()
score = grp.groupby('model', as_index=False)['abs_error_pp'].mean().sort_values('abs_error_pp')
score.to_csv(OUT/'group_mae.csv', index=False)
grp.to_csv(OUT/'group_predictions.csv', index=False)
print('\nGrouped success-rate MAE (%p):\n', score.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(score['model'], score['abs_error_pp'])
ax.set_ylabel('Group MAE (percentage points)')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(OUT/'model_comparison.png', dpi=180)
plt.show()
print('\nSaved results to:', OUT.resolve())


## 2. 결과 확인 및 다운로드
다변수 모델의 검증 성능표·그래프를 확인하고 ZIP 파일로 내려받습니다. 기존 실험의 `payload_split.csv`를 사용하므로 데이터 누출을 막을 수 있습니다.

In [ ]:
from IPython.display import display, Image
from pathlib import Path
import shutil
display(metrics)
display(score)
display(Image(filename=str(OUT/'model_comparison.png')))
archive = shutil.make_archive('qrculus_multivariate_results', 'zip', root_dir='.', base_dir=str(OUT))
print('ZIP:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Colab 밖에서는 생성된 ZIP 파일을 직접 내려받으세요.')